# AI Terapeut — Reinforcement Learning walkthrough

Tento notebook ukazuje, jak v této úloze funguje reinforcement learning:
- jak vypadá stav a akce,
- jak funguje rule-based baseline,
- jak trénujeme tabulární Q-learning,
- jak porovnat baseline vs RL a vizualizovat výsledky.

## 1) Instalace a importy
Pokud běžíte v Colabu, odkomentujte instalační buňku.

In [ ]:
# !pip install -q -r requirements.txt

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rehab_solution import (
    RehabEnv,
    PATIENT_PROFILES,
    train_q_learning,
    run_policy,
    rule_based_policy,
    q_policy,
    write_predictions_csv,
)

plt.style.use('seaborn-v0_8')

## 2) Jak vypadá prostředí
Stav = `[recovery, pain, fatigue, motivation, day_normalized]`, akce = 0..4 (`rest` až `functional`).

In [ ]:
env = RehabEnv(seed=42)
obs, info = env.reset(profile_name='balanced', seed=42)
print('Profiles:', list(PATIENT_PROFILES.keys()))
print('Action space:', env.action_space)
print('Observation space:', env.observation_space)
print('Initial observation:', obs)
print('Reset info:', info)

## 3) Trénink RL agenta (Q-learning)
Pro rychlejší demo můžete snížit `episodes`.

In [ ]:
env = RehabEnv(seed=42)
q_table = train_q_learning(env, episodes=3000, seed=42)
print('Q-table states:', len(q_table))

## 4) Evaluace baseline vs RL
Stejný setup pro obě politiky (všech 4 profilech, stejné seedování).

In [ ]:
env = RehabEnv(seed=42)
rule_rows, rule_summary = run_policy(
    env,
    policy_name='rule_based',
    policy_fn=lambda obs, profile: rule_based_policy(obs, profile),
    episodes_per_profile=25,
    seed=2026,
)

rl_rows, rl_summary = run_policy(
    env,
    policy_name='q_learning',
    policy_fn=lambda obs, profile: q_policy(obs, q_table),
    episodes_per_profile=25,
    seed=2026,
)

display(pd.DataFrame([
    {
        'policy': 'rule_based',
        'avg_total_reward': rule_summary['avg_total_reward'],
        'avg_final_recovery': rule_summary['avg_final_recovery'],
        'avg_final_pain': rule_summary['avg_final_pain'],
        'success_rate': rule_summary['success_rate'],
    },
    {
        'policy': 'q_learning',
        'avg_total_reward': rl_summary['avg_total_reward'],
        'avg_final_recovery': rl_summary['avg_final_recovery'],
        'avg_final_pain': rl_summary['avg_final_pain'],
        'success_rate': rl_summary['success_rate'],
    },
]))

## 5) Vizualizace výsledků
### 5.1 Final recovery a pain podle profilů

In [ ]:
rule_df = pd.DataFrame(rule_rows)
rl_df = pd.DataFrame(rl_rows)

rule_last = rule_df.sort_values(['profile', 'episode', 'day']).groupby(['profile', 'episode']).tail(1)
rl_last = rl_df.sort_values(['profile', 'episode', 'day']).groupby(['profile', 'episode']).tail(1)

agg = pd.concat([
    rule_last.assign(policy='rule_based'),
    rl_last.assign(policy='q_learning'),
])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for i, metric in enumerate(['recovery', 'pain']):
    pivot = agg.groupby(['profile', 'policy'])[metric].mean().unstack()
    pivot.plot(kind='bar', ax=axes[i], rot=20)
    axes[i].set_title(f'Final {metric} by profile')
    axes[i].set_ylabel(metric)

plt.tight_layout()
plt.show()

### 5.2 Distribuce vybraných akcí

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

(rule_df['action_name'].value_counts(normalize=True).sort_index() * 100).plot(kind='bar', ax=axes[0])
axes[0].set_title('Rule-based action distribution (%)')
axes[0].set_ylabel('Percent')

(rl_df['action_name'].value_counts(normalize=True).sort_index() * 100).plot(kind='bar', ax=axes[1])
axes[1].set_title('Q-learning action distribution (%)')
axes[1].set_ylabel('Percent')

plt.tight_layout()
plt.show()

## 6) Uložení CSV predikcí
Tato buňka zapíše stejné CSV soubory jako hlavní skript.

In [ ]:
out_dir = Path('.')
write_predictions_csv(out_dir / 'rulebased_predictions.csv', rule_rows)
write_predictions_csv(out_dir / 'rl_predictions.csv', rl_rows)
print('Saved:', out_dir / 'rulebased_predictions.csv')
print('Saved:', out_dir / 'rl_predictions.csv')